In [1]:
from datasets import load_dataset
from angle_emb import AnglE, AngleDataTokenizer

import pandas as pd

import shutil
import os

from datasets import Dataset
from huggingface_hub import login

import model_utils
import utils

In [2]:

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Use GPU 0
os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["XET_NO_PROGRESS"] = "true"

# Disable dataset progress bars
from datasets import disable_progress_bars
disable_progress_bars()


In [3]:
data_dir = '/home/shay/PycharmProjects/sensim/data'

df_mapping = {'SentenceA': 'text1', 'SentenceB': 'text2', 'score': 'label'}
# df_columns = ['SentenceA', 'SentenceB', 'score']
df_columns = df_mapping.keys()

def get_dataset(filename):
    df = pd.read_csv(os.path.join(data_dir, filename), sep='\t')
    df = df[df_columns]
    df.rename(columns=df_mapping, inplace=True)
    df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)
    dataset = Dataset.from_pandas(df_shuffled)
    return dataset

train_dataset = get_dataset('llms_pairs_2500_scored.csv')
#train_dataset = train_dataset.select(range(2000))
validation_dataset = get_dataset('validation_pairs_100_scored.csv')
test_dataset = get_dataset('test_pairs_300_scored.csv')
#test_dataset = get_dataset('validation_pairs_100_scored.csv')



In [4]:

base_model_name = 'OMRIDRORI/mbert-tibetan-continual-wylie-final'
#base_model_name = 'Xenova/all-mpnet-base-v2'
hf_gated_omri = ''

login(token=hf_gated_omri)

# 1. load pretrained model
angle = AnglE.from_pretrained(base_model_name, token=hf_gated_omri, max_length=512, pooling_strategy='cls').cuda()
print('Model loaded')
# 3. transform data
train_ds = train_dataset.map(AngleDataTokenizer(angle.tokenizer, angle.max_length), num_proc=8)
valid_ds = validation_dataset.map(AngleDataTokenizer(angle.tokenizer, angle.max_length), num_proc=8)
test_ds = test_dataset.map(AngleDataTokenizer(angle.tokenizer, angle.max_length), num_proc=8)
# train_ds = train_dataset.map(AngleDataTokenizer(angle.tokenizer, angle.max_length))
# valid_ds = validation_dataset.map(AngleDataTokenizer(angle.tokenizer, angle.max_length))
# test_ds = test_dataset.map(AngleDataTokenizer(angle.tokenizer, angle.max_length))
print('datasets loaded')

Some weights of BertModel were not initialized from the model checkpoint at OMRIDRORI/mbert-tibetan-continual-wylie-final and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded


INFO:AnglE:Detect DatasetFormats.A: text1,text2,label
INFO:AnglE:Detect DatasetFormats.A: text1,text2,label
INFO:AnglE:Detect DatasetFormats.A: text1,text2,label
INFO:AnglE:Detect DatasetFormats.A: text1,text2,label
INFO:AnglE:Detect DatasetFormats.A: text1,text2,label
INFO:AnglE:Detect DatasetFormats.A: text1,text2,label
INFO:AnglE:Detect DatasetFormats.A: text1,text2,label
INFO:AnglE:Detect DatasetFormats.A: text1,text2,label
INFO:AnglE:Detect DatasetFormats.A: text1,text2,label
INFO:AnglE:Detect DatasetFormats.A: text1,text2,label
INFO:AnglE:Detect DatasetFormats.A: text1,text2,label
INFO:AnglE:Detect DatasetFormats.A: text1,text2,label
INFO:AnglE:Detect DatasetFormats.A: text1,text2,label
INFO:AnglE:Detect DatasetFormats.A: text1,text2,label
INFO:AnglE:Detect DatasetFormats.A: text1,text2,label
INFO:AnglE:Detect DatasetFormats.A: text1,text2,label
INFO:AnglE:Detect DatasetFormats.A: text1,text2,label
INFO:AnglE:Detect DatasetFormats.A: text1,text2,label
INFO:AnglE:Detect DatasetFor

datasets loaded


In [5]:
# Delete checkpoint folder if it exists
if os.path.exists('ckpts/sts-b'):
    shutil.rmtree('ckpts/sts-b')

# training_args = TrainingArguments(
#     output_dir='ckpts/sts-b',
#     num_train_epochs=5,
#     learning_rate=2e-5,
#     per_device_train_batch_size=32,
#     gradient_accumulation_steps=16,
#     warmup_steps=0,
#     save_steps=100,
#     eval_steps=1000,           # Note: The argument is 'eval_steps', not 'evaluation_strategy' with steps
#     logging_steps=100,
#     fp16=True,
#     report_to="none",          # This is the correct way to disable wandb logging
# )

loss_kwargs={
    'cosine_w': 1e-2, #0.0,
    'ibn_w': 1.0,
    'cln_w': 1.0,
    'angle_w': 0.02,
    'cosine_tau': 20,
    'ibn_tau': 20,
    'angle_tau': 20
}

# 4. fit
angle.fit(
    train_ds=train_ds,
    valid_ds=valid_ds,
    output_dir='ckpts/sts-b',
    batch_size=32,
    epochs=5,
    learning_rate=2e-5,
    save_steps=100,
    eval_steps=1000,
    warmup_steps=0,
    gradient_accumulation_steps=16, #1,
    loss_kwargs=loss_kwargs,
    fp16=True,
    logging_steps=100,
)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/home/shay/PycharmProjects/sensim/.venv/lib/python3.10/site-packages/angle_emb/angle.py:766: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `AngleTrainer.__init__`. Use `processing_class` instead.
  super().__init__(**kwargs)
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 3}.
You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss,Validation Loss


In [6]:

# 5. evaluate
print('evaluating')

train_spearman_correlation = model_utils.bi_encoder_evaluate(train_ds, metric ='spearman_cosine')
print('Train set spearman_cosine spearman_correlation:', train_spearman_correlation)
train_pearson_correlation = model_utils.bi_encoder_evaluate(train_ds, metric ='pearson_cosine')
print('Train set  pearson_cosine pearson_correlation:', train_pearson_correlation)

valid_spearman_correlation = model_utils.bi_encoder_evaluate(valid_ds, metric ='spearman_cosine')
print('Validation set spearman_cosine spearman_correlation:', valid_spearman_correlation)
valid_pearson_correlation = model_utils.bi_encoder_evaluate(valid_ds, metric ='pearson_cosine')
print('Validation set  pearson_cosine pearson_correlation:', valid_pearson_correlation)

test_spearman_correlation = model_utils.bi_encoder_evaluate(test_ds, metric ='spearman_cosine')
print('Test set spearman_cosine spearman_correlation:', test_spearman_correlation)
test_pearson_correlation = model_utils.bi_encoder_evaluate(test_ds, metric ='pearson_cosine')
print('Test set pearson_cosine pearson_correlation:', test_pearson_correlation)

print('done evaluating')

eval_result = {
    "train_spearman": train_spearman_correlation, "train_pearson": train_pearson_correlation,
    "valid_spearman": valid_spearman_correlation, "valid_pearson": valid_pearson_correlation,
    "test_spearman": test_spearman_correlation, "test_pearson": test_pearson_correlation,
}

run_settings = {"model": "AnglE", "pretrained": base_model_name}
run_settings.update(loss_kwargs)
log_results_file = './results/sensim_results.csv'
utils.log_evaluation_results(log_file=log_results_file, settings=run_settings, results=eval_result)


evaluating


79it [00:04, 17.48it/s]                        


Train set spearman_cosine spearman_correlation: 0.7401978131566748


79it [00:04, 17.62it/s]                        


Train set  pearson_cosine pearson_correlation: 0.7065498785103066


...

Validation set  pearson_cosine pearson_correlation: 0.7510527419376197


10it [00:00, 12.29it/s]                      


Test set spearman_cosine spearman_correlation: 0.7747266997304972


10it [00:00, 12.45it/s]                      


Test set pearson_cosine pearson_correlation: 0.7336620678225586
done evaluating
New entry added to log for settings: {'model': 'AnglE', 'base model': 'OMRIDRORI/mbert-tibetan-continual-wylie-final', 'cosine_w': 0.01, 'ibn_w': 1.0, 'cln_w': 1.0, 'angle_w': 0.02, 'cosine_tau': 20, 'ibn_tau': 20, 'angle_tau': 20}
